# Демо SoundStream

Запустите все ячейки. Укажите `AUDIO_URL`.

## 1. Ссылка на аудио

In [ ]:
AUDIO_URL = "https://keithito.com/LJ-Speech-Dataset/LJ025-0076.wav"

## 2. Клонирование репозитория

In [ ]:
!git clone --depth 1 https://github.com/9imon4ik/soundstream-neural-audio-codec.git

In [ ]:
%cd soundstream-neural-audio-codec

## 3. Установка зависимостей

In [ ]:
%pip install -q hydra-core omegaconf huggingface_hub requests soundfile

## 4. Модель

In [ ]:
from pathlib import Path

import torch
from huggingface_hub import hf_hub_download
from hydra import compose, initialize_config_dir
from hydra.utils import instantiate

device = "cuda" if torch.cuda.is_available() else "cpu"

hf_hub_download(
    repo_id="9imon4ik/soundstream-neural-audio-codec",
    filename="checkpoint-epoch100.pth",
    local_dir="saved/soundstream-final",
)

config_dir = str((Path.cwd() / "src/configs").resolve())
with initialize_config_dir(version_base=None, config_dir=config_dir):
    cfg = compose(config_name="soundstream")

generator = instantiate(cfg.generator).to(device).eval()
generator.load_state_dict(
    torch.load(
        "saved/soundstream-final/checkpoint-epoch100.pth",
        map_location=device,
        weights_only=False,
    )["generator_state_dict"]
)

## 5. Аудио

In [ ]:
import io

import requests
import torchaudio

wav, sr = torchaudio.load(io.BytesIO(requests.get(AUDIO_URL).content))
audio = torchaudio.functional.resample(wav.mean(0, keepdim=True), sr, 16000)

## 6. Ресинтез

In [ ]:
with torch.no_grad():
    reconstructed = generator(audio.unsqueeze(0).to(device))["reconstructed_audio"].squeeze().cpu()

## 7. Прослушивание

In [ ]:
from IPython.display import Audio, display

display(Audio(audio.squeeze().numpy(), rate=16000))
display(Audio(reconstructed.numpy(), rate=16000))